# Elo Modeling

Tujuan : 

Notebook ini bertujuan mengevaluasi pengaruh penambahan **Elo Rating** terhadap performa model dalam memprediksi hasil pertandingan sepak bola internasional.

Berbeda dengan *Historical Modeling* yang hanya memanfaatkan fitur dasar dan statistik historis, tahap ini menambahkan representasi kekuatan relatif tim melalui **Elo Rating** yang dihitung secara kronologis menggunakan seluruh riwayat pertandingan.

Eksperimen tetap menggunakan algoritma dan strategi pemodelan yang sama, yaitu **Logistic Regression** dan **Random Forest**, sehingga setiap perubahan performa dapat diatribusikan pada informasi tambahan yang diberikan oleh fitur Elo, bukan karena perubahan model atau proses *preprocessing*.

Hasil evaluasi pada notebook ini akan dibandingkan dengan eksperimen sebelumnya untuk mengetahui apakah Elo Rating mampu meningkatkan kualitas prediksi secara terukur.

## 1. Load Elo Dataset

Dataset yang digunakan pada tahap ini merupakan hasil dari notebook **Elo Rating Engine**. Dataset tersebut telah memuat seluruh fitur dasar (*baseline features*), fitur historis (*historical features*), serta fitur **Elo Rating** yang dibangun secara kronologis.
Sebelum memulai proses pemodelan, kita akan memuat dataset dan melakukan pengecekan singkat untuk memastikan struktur data sesuai dengan yang diharapkan.

In [1]:
import pandas as pd

In [2]:
elo_df = pd.read_csv("../data/processed/elo_features.csv")

In [3]:
elo_df.head()

,date,home_team,away_team,home_score,away_score,home_elo_before,away_elo_before,home_elo_after,away_elo_after
0,1872-11-30,Scotland,England,0.0,0.0,1500.000000,1500.000000,1500.000000,1500.000000
1,1873-03-08,England,Scotland,4.0,2.0,1500.000000,1500.000000,1510.000000,1490.000000
2,1874-03-07,Scotland,England,2.0,1.0,1490.000000,1510.000000,1500.575011,1499.424989
3,1875-03-06,England,Scotland,2.0,2.0,1499.424989,1500.575011,1499.458089,1500.541911
4,1876-03-04,Scotland,England,3.0,0.0,1500.541911,1499.458089,1510.510716,1489.489284


In [4]:
elo_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49433 entries, 0 to 49432
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             49433 non-null  str    
 1   home_team        49433 non-null  str    
 2   away_team        49433 non-null  str    
 3   home_score       49433 non-null  float64
 4   away_score       49433 non-null  float64
 5   home_elo_before  49433 non-null  float64
 6   away_elo_before  49433 non-null  float64
 7   home_elo_after   49433 non-null  float64
 8   away_elo_after   49433 non-null  float64
dtypes: float64(6), str(3)
memory usage: 3.4 MB


Dataset berhasil dimuat dan telah memuat seluruh fitur yang dibutuhkan untuk proses pemodelan. Selain fitur dasar dan fitur historis, dataset kini juga memiliki empat kolom Elo Rating, yaitu:

- `home_elo_before`
- `away_elo_before`
- `home_elo_after`
- `away_elo_after`

Pada tahap pemodelan, hanya **Elo sebelum pertandingan** (`home_elo_before` dan `away_elo_before`) yang akan digunakan sebagai fitur prediktif karena nilai tersebut memang tersedia sebelum pertandingan dimulai. Sebaliknya, `home_elo_after` dan `away_elo_after` merupakan hasil pembaruan setelah pertandingan selesai sehingga tidak digunakan sebagai input model untuk menghindari *future data leakage*.